# Soro-Kev UPP: P0 architecture test + parent check (2x T4)

Runs the Universal Probabilistic Pointer ablation grid and checks parent backbones. It decides the
architecture and the parent; it is not the final model.

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **on**.
Expected time on 2x T4: arch bench ~10 min, parent check ~15 min, P0 grid ~3 h (both GPUs), Gemma A/J ~1 h.

| ID | Variant | Question it answers |
| :-- | :-- | :-- |
| A | Kev dot pointer (query first) | baseline |
| F | Block-isolated options, full text visible | does exact order invariance cost accuracy? |
| G | Distance-aware (spectral norm + RBF) head | better calibration / abstain? |
| H | 4-head ensemble + energy | better calibration / abstain? |
| P | Block + attention-pooled option readout | better option representation? |
| K | Block + 4 gist tokens per option | can options be compressed to 4 cached tokens? |
| K8 | Block + 8 gists | capacity vs compression |
| Ksd | K + gist self-distillation | does matching the full-text path recover accuracy? |
| L | Late interaction (MaxSim over gists, no cross-attention) | retriever-cost end of the spectrum |
| J2 | Ksd + RBF head + 4-head ensemble | candidate final head |
| KM | 8 Matryoshka gists + distillation; also scored by budget (1/2/4/8) and two-level (1 gist x 256 -> top 16 x 8 gists) | one cache for retrieval + rerank |

Every variant is scored on held-out Soro packages, the zero-shot split, BFCL, and ToolRet
(600 queries with MiniLM top-16 menus, 300 with top-256 menus for models that can read them; first-stage recall is reported separately: MiniLM recall@16 = 47.7%, @256 = 83.3%, @1024 = 91.0%).

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/nisaral/Soro-Boro.git"
BRANCH = "upp-p0"          # branch with veyra/python/src/soro/routing/upp.py
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
REPO = f"{WORK}/Soro-Boro"
# Option B: upload the repo as a Kaggle dataset and set LOCAL_COPY to its path.
LOCAL_COPY = None  # e.g. "/kaggle/input/soro-boro/Soro-Boro"
if LOCAL_COPY:
    subprocess.run(["cp", "-r", LOCAL_COPY, REPO], check=True)
elif not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
!pip -q install -U "transformers>=5.0" peft bitsandbytes accelerate huggingface_hub safetensors
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, transformers, peft
print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.device_count(), "GPU(s)")

## 0. Architecture bench (inference only): exactness of block / gist / composed caches and latency vs menu size

In [ ]:
!python veyra/training/arch_bench.py --base-model Qwen/Qwen3-0.6B-Base --sizes 6,16,64,256,1024 --out runs/arch_bench_qwen3_0.6b.json

## 1. Parent check
Block isolation exactness, cache exactness, early exits, VRAM and latency for each candidate.
Qwen3.5 is left out on purpose: its Gated DeltaNet layers cannot run the block layout.

In [ ]:
!python veyra/training/parent_check.py   --models Qwen/Qwen3-0.6B-Base,Qwen/Qwen3-1.7B-Base,google/gemma-4-E2B   --four-bit google/gemma-4-E2B   --out runs/parent_check.json

## 2. P0 grid on Qwen3-0.6B-Base (both GPUs in parallel)
Same data, seed and steps for every variant. Held-out: train packages carved out for selection,
and a stride sample of `val_zeroshot.jsonl`. Data comes from the public HF dataset.

In [ ]:
COMMON = "--base-model Qwen/Qwen3-0.6B-Base --train-rows 8000 --eval-rows 400 --batch 8 --max-seq-len 832 --log-every 50"
gpu0 = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 python veyra/training/p0_upp_ablation.py --configs A,F,H,J {COMMON} --out runs/p0_gpu0 > runs_p0_gpu0.log 2>&1", shell=True)
gpu1 = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 python veyra/training/p0_upp_ablation.py --configs B,G,I {COMMON} --out runs/p0_gpu1 > runs_p0_gpu1.log 2>&1", shell=True)
print("started; tail the logs in the next cell")

In [ ]:
!tail -n 3 runs_p0_gpu0.log; echo ---; tail -n 3 runs_p0_gpu1.log

In [ ]:
gpu0.wait(); gpu1.wait()
import json
res = {}
for d in ("runs/p0_gpu0", "runs/p0_gpu1"):
    res.update(json.load(open(f"{d}/results.json")))
json.dump(res, open("runs/p0_qwen3_0.6b.json", "w"), indent=2)
sys.path.insert(0, "veyra/training")
from p0_upp_ablation import markdown
md_table = markdown(dict(sorted(res.items())))
open("runs/p0_qwen3_0.6b.md", "w").write(md_table)
print(md_table)

## 3. Parent run: Gemma 4 E2B (4-bit), baseline vs combined
Confirms the architecture transfers to the planned omni parent. Text only here.

In [ ]:
%%bash
# Baseline (A) and combined (J) on Gemma 4 E2B, one per T4, in parallel.
# 4-bit weights; Gemma computes in fp32 on T4 (fp16 overflows in Gemma activations).
for spec in "0 A" "1 J"; do set -- $spec
  CUDA_VISIBLE_DEVICES=$1 python veyra/training/p0_upp_ablation.py --base-model google/gemma-4-E2B --load-in-4bit \
    --configs $2 --train-rows 4000 --eval-rows 400 --batch 4 --max-seq-len 832 --log-every 50 \
    --out runs/p0_gemma4_e2b_$2 > gemma_$2.log 2>&1 &
done
wait
tail -n 4 gemma_A.log gemma_J.log


In [ ]:
!zip -qr /kaggle/working/p0_results.zip runs runs_p0_gpu0.log runs_p0_gpu1.log gemma_A.log gemma_J.log 2>/dev/null || zip -qr p0_results.zip runs
print("Download p0_results.zip from the output panel and send it back.")